### Based on Shi et al. 2019

In [1]:
# Package loading
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import numpy as np
from sklearn.model_selection import train_test_split
import pandas as pd

keras.utils.set_random_seed(42)

In [2]:
all_images = []
img_directory = 'molecule_images/'

# Runs from 1 to 1547(the greatest numbered img) and will throw errors but continue on missing file names.
for i in range(1, 1548):
    try:
        img = tf.io.read_file('molecule_images/BACE_' + str(i) + '.png')
        all_images.append(tf.io.decode_png(img))
    except:
        pass

In [3]:
# Extract target variable
bace_data = pd.read_csv('bace.csv')
pIC = bace_data['pIC50']

# Split into training and test/validation sets
X_train, X_tval, y_train, y_tval = train_test_split(all_images, pIC, test_size = 0.2, random_state = 42)

# Split test and validation sets
X_val, X_test, y_val, y_test = train_test_split(X_tval, y_tval, test_size = 0.5, random_state = 42)

# Convert X's to tensors to avoid later errors
X_train = tf.convert_to_tensor(X_train)
X_test = tf.convert_to_tensor(X_test)
X_val = tf.convert_to_tensor(X_val)

In [5]:
# Construct model
inputs = keras.Input(shape = (400, 400, 3))
# Original model used 300x300 images but we don't have the compute for that
x = layers.Resizing(180, 180)(inputs)
x = layers.Rescaling(1./255)(x)
x = layers.Conv2D(filters = 16, kernel_size = (21, 21), activation = "relu")(x)
x = layers.MaxPool2D(pool_size = 14, padding = "same")(x)
x = layers.Flatten()(x)
x = layers.Dense(512, activation = "relu")(x)
x = layers.Dropout(0.4)(x)

outputs = layers.Dense(1, activation = "linear")(x)

model = keras.Model(inputs = inputs, outputs = outputs)

model.compile(loss = "mse",
              optimizer = "adam",
              metrics = ["mae"])

In [6]:
model.fit(X_train, y_train, epochs = 100, validation_data = (X_val, y_val), batch_size = 128)

Epoch 1/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 28s 1s/step - loss: 106.3562 - mae: 6.6361 - val_loss: 6.8593 - val_mae: 2.3335
Epoch 2/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 170ms/step - loss: 6.4151 - mae: 2.1140 - val_loss: 3.2288 - val_mae: 1.5625
Epoch 3/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 158ms/step - loss: 3.2600 - mae: 1.4820 - val_loss: 2.2967 - val_mae: 1.3082
Epoch 4/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 158ms/step - loss: 2.4016 - mae: 1.2733 - val_loss: 1.8948 - val_mae: 1.1631
Epoch 5/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 165ms/step - loss: 2.1736 - mae: 1.2124 - val_loss: 1.7996 - val_mae: 1.0880
Epoch 6/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 168ms/step - loss: 2.1735 - mae: 1.2122 - val_loss: 1.9880 - val_mae: 1.0982
Epoch 7/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 158ms/step - loss: 2.1486 - mae: 1.1986 - val_loss: 1.9646 - val_mae: 1.0946
Epoch 8/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 158ms/step - loss: 2.1676 - mae: 1.1973 - val_loss: 1.7864 - val_mae: 1.1047
Epoch 9/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 159ms/